# Particle Filter

Das Ziel des hier implementierten Particle Filters ist es, die Position des Smartphones zum Zeitpunkt $k$ zu bestimmen.<br>
Eingabedaten kommen dabei zum einen aus dem Pedestrian dead Reckoning, zum anderen aus der Trilateration der RSSI Daten. Zum fusionieren der beiden Datenquellen kann sowohl ein Kalman-, als auch ein Particle Filter verwendet werden. Letzterer eignet sich aus mehreren Gründen besonders für den gegebenen Anwendungsfall und wird deshalb implementiert:
- Durch die Lokalisierung im Innenbereich eines Gebäudes liegen natürliche Restriktionen des möglichen Pfads vor (Wände). Diese können beim Particle-Filter gut modelliert werden, indem man verhindert, dass die einzelnen Partikel zwischen den Zeitpunkten $k$ und $k-1$ eine Wand kreuzen.
- Die Fehlerverteilung von RSSI-Signalen entspricht aufgrund von Abschattungen und Mehrwegeausbreitung in Innenräumen eher einer PDF (Propability-Density-Function) als einer gausschen Normalverteilung.

## Import statements

In [1]:
import sqlite3
import time

import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
from IPython.display import clear_output

import math
import pandas as pd
import numpy as np

## Read data

In [2]:
conn = sqlite3.connect("../data/emi_nav.db")

run_id =  "R3"

df_steps = pd.read_sql(
    f"""
        SELECT * FROM steps
        WHERE run_id = '{run_id}'
        ORDER BY timestamp_ms
    """, conn)

df_heading = pd.read_sql(
    f"""
        SELECT * FROM heading
        WHERE run_id = '{run_id}'
        ORDER BY timestamp_ms
    """, conn)


df_steps['t_sec'] = (df_steps.timestamp_ms - df_steps.timestamp_ms.min()) / 1000
df_heading['t_sec'] = (df_heading.timestamp_ms - df_heading.timestamp_ms.min()) / 1000

In [3]:
map_eg = np.load(r'..\data\floorplans\eg.npy', allow_pickle=True)
map_og1 = np.load(r'..\data\floorplans\og1.npy', allow_pickle=True)

## Kombination von df_steps und df_heading
Bei jedem Schritt wird ein Prediktionsschritt im Filter ausgelöst mit dem Heading zum Zeitpunkt des Schrittes und einer festgelegten Schrittlänge.

In [4]:
df_steps = df_steps.drop(columns='heading_rad', errors='ignore') # drop heading_rad if already exists

df_steps = pd.merge_asof(
    df_steps,
    df_heading[['t_sec', 'heading_rad']],
    on='t_sec',
    direction='nearest'
)[['t_sec', 'heading_rad']]
df_steps['dt'] = np.diff(df_steps.t_sec, prepend=0.0)
df_steps.head(5)

,t_sec,heading_rad,dt
0,0.000,0.608923,0.000
1,0.408,0.522946,0.408
2,1.080,0.400481,0.672
3,1.616,1.225383,0.536
4,2.233,1.357017,0.617


## Plot
Hier wird die Funktion zum Plotten des aktuellen States des Particle Filters implementiert.

In [22]:
H, W = map_eg.shape

def plot_map_particles(occupancy_map, particles, resolution=0.2):
    fig, ax = plt.subplots(figsize=(14, 14 * H / W + 1))

    # Map: extent in Metern, origin='lower' = row 0 unten (wie beim Plotly-Heatmap)
    ax.imshow(
        occupancy_map,
        cmap='gray',                      # 0 -> schwarz, 1 -> weiß (wie eure colorscale)
        origin='lower',
        extent=[0, W * resolution, 0, H * resolution],
        interpolation='nearest',
    )

    # Partikel
    sc = ax.scatter(
        particles[:, 0], particles[:, 1],
        s=3,
        c=particles[:, 2],
        cmap='viridis',
    )
    fig.colorbar(sc, ax=ax, label='Gewicht', shrink=0.6)

    ax.set_aspect('equal')
    ax.set_xlabel('x [m]')
    ax.set_ylabel('y [m]')
    ax.set_title(f'{len(particles)} Partikel')
    plt.tight_layout()
    plt.show()

## Particle Filter
### Initialisierung

Zu Beginn werden $m$ Partikel zufällig per Gleichverteilung in der Karte initialisiert

In [17]:
def initialize_particles(occupancy_map, m=300, resolution=0.2):
    """
    Initializes m particles inside the occupancy map.
    :param occupancy_map: A binary map representing the building structure.
    :param m: The amount of particles to initialize.
    :param resolution: The resolution of the map.
    :returns: A mx2 matrix with the coordinates (x,y) of m particles.
    """
    free = np.argwhere(occupancy_map == 1)
    idx = np.random.default_rng().integers(0, len(free), m)
    cells = free[idx]

    x = (cells[:, 1] + np.random.default_rng().random(m)) * resolution
    y = (cells[:, 0] + np.random.default_rng().random(m)) * resolution

    particles = np.column_stack([x, y, np.ones((m,))])
    
    return particles

## Prediktions-Schritt
Im Prädiktionsschritt werden die Partikel basierend auf der PDR um einen Schritt Richtung des gemessenen Heading bewegt. Dabei wird ein Normalverteilter Fehler sowohl dem Heading als auch der Distanz hinzugefügt.

In [44]:
def move(particles, d, heading, sigma_d, sigma_heading):
    """
    Moves each particle using the given distance d and heading with a normal ditributed error N(0,sigma_d) and N(0, sigma_heading) applied.
    :param particles: A mx3 matrix with m particles.
    :param d: The distance to move the pixels in m.
    :param heading: The direction of movement in rad.
    :param sigma_d: The variance of the error distribution of the distance.
    :param sigma_heading: The variance of the error distribution of the heading.
    :returns: A mx3 matrix with m updated particles.
    """
    m, _ = particles.shape
    
    # Calculate the directional and orientational errors
    epsilon_d = np.random.default_rng().normal(0, sigma_d, m)
    epsilon_heading = np.random.default_rng().normal(0, sigma_heading, m)
    
    particles_moved = particles.copy()
    
    # apply motion model
    particles_moved[:, 0] += np.cos(heading + epsilon_heading) * (d + epsilon_d)
    particles_moved[:, 1] += np.sin(heading + epsilon_heading) * (d + epsilon_d)
    
    return particles_moved

def update_weights(old_particles, particles, occupancy_map, resolution=0.2):
    """
    Checks for each particle, if the line between the old and the new one passes a wall in the map.
    If a particle passes a wall, its weight will be set to 0.
    :param old_particles: A mx3 matrix of m particle coordinates in pixels at timestep k-1.
    :param particles: A mx3 matrix of m particle coordinates in pixels at timestep k.
    :param occupancy_map: A binary map representing walls and walkable areas.
    :returns: A mx3 matrix of m particles with new weights applied.
    """
    height, width = occupancy_map.shape
    p0 = old_particles[:, :2] / resolution
    p1 = particles[:, :2] / resolution
    d = p1 - p0 
    seg_len = np.linalg.norm(d, axis=1)
    n_samples = max(2, int(np.ceil(seg_len.max() / 0.5)) + 1)    
    t = np.linspace(0.0, 1.0, n_samples)                  # (S,)

    pts = p0[:, None, :] + d[:, None, :] * t[None, :, None]   # (m, S, 2)
    cols = np.round(pts[..., 0]).astype(int)              # x -> col
    rows = np.round(pts[..., 1]).astype(int)              # y -> row

    # Out-of-bounds zählt als Wandtreffer
    oob = (rows < 0) | (rows >= H) | (cols < 0) | (cols >= W)
    r = np.clip(rows, 0, H - 1)
    c = np.clip(cols, 0, W - 1)

    hit = (occupancy_map[r, c] == 0) | oob       # (m, S)
    crossed = hit.any(axis=1)                             # (m,)

    particles[crossed, 2] = 0.0
    
    return particles

In [45]:
steps = len(df_steps)
d=0.7

step = 0
speed = 1.0

particles = initialize_particles(map_eg, m=300)

for _, row in df_steps.iterrows():
        t0 = time.perf_counter()

        old_particles = particles.copy()
        particles = move(particles, d, df_steps.iloc[step].heading_rad + math.pi/2, 0.07, 0.05)
        particles = update_weights(old_particles, particles, map_eg)

        clear_output(wait=True)
        plot_map_particles(map_eg, particles)

        elapsed = time.perf_counter() - t0
        wait = row['dt'] / speed - elapsed
        if wait > 0:
            time.sleep(wait)
            
        step += 1

KeyboardInterrupt: 